# 0.4秒 BC · 全量数据 · 卡牌落点先验 + CNN64

沿用已发布的 **10,654局 / 333源分片**，把最新完整BC checkpoint的实体编码、时间主干和选牌网络保留下来并冻结，单独训练经过本地试验的新落点后端。新的先验统计只来自固定训练划分；验证和heldout不参与统计或训练。

- 新阶段单独保存checkpoint、优化器、进度和W&B身份，不接着旧Adam训练。父阶段9,887步只记录为来源，不计入本阶段进度。
- 默认 **8 epoch**；先以 **3e-4** 训练落点头，完成4次定期保存（2048个训练视角）后自动全量解冻，统一学习率 **1e-5**。保留落点头Adam，新增主干Adam按需初始化。训练温度1。
- 保持8个轨迹座位并行、frame microbatch256、CUDA FP16及已有数值保护。每局流式解压后回收，只从Drive缓存必要数据包和一个父checkpoint到本地SSD。
- 保持4%固定验证池（约426局）、每轮随机16局双视角验证、5局独立heldout、不早停。验证指标只用于记录和挑选best，不能替代完整实战评测。
- `latest.pt`完整恢复同一新阶段；`best.pt`用于评测。训练统计、模型定义和新后端权重随checkpoint保存。
- **按顺序运行6个代码单元格；代码5才启动训练。** CPU准备不消耗GPU；在Colab切换L4后从代码1开始。中断显示后只重跑代码5连接后台任务。
- 保持0.4秒决策及现有动作保留数据，不加入WAIT/攒费战术规则，不更改固定验证划分。


## 代码1 / 6：挂载Drive并设置新后端阶段

数据包沿用CPU已发布的全量数据。新阶段使用独立的Drive目录与`/content`缓存，不覆盖旧训练。


In [ ]:
from pathlib import Path
import os, sys, json, time
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
SOURCE_RUN = Path('/content/drive/MyDrive/cr_rl/runs/state_global_event040_20261110')
RELEASE = SOURCE_RUN / 'bc_full_20261111'
NOTEBOOK_RUN = RELEASE / 'experiment_placement_priorcnn64_20261111'
PACK_INDEX = RELEASE / 'data_pack/current.json'
WORK = Path('/content/crrl_bc_full_priorcnn64_20261111')
TARGET_EPOCHS = 36
MIN_TRAIN_GAMES = 1000
EXPECTED_TOTAL_GAMES = 10654
EPISODE_BATCH_SIZE = 16
FRAME_MICROBATCH = 512
VALIDATION_FRACTION = 0.04
VALIDATION_GAMES = 16
EARLY_STOPPING_PATIENCE = 0
EARLY_STOPPING_MIN_DELTA = 0.01
LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 3e-4
WARMUP_STEPS = 0
LR_SCHEDULE = 'constant'
WANDB_PROJECT = 'cr-rl'
WANDB_ENTITY = None
WANDB_NAME = 'bc040-183m-priorcnn64-joint36ep'
WORK.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RUN.mkdir(parents=True, exist_ok=True)
print('CPU发布的数据包索引:', PACK_INDEX)
print('新实验:', NOTEBOOK_RUN)
print('完整数据要求:', EXPECTED_TOTAL_GAMES, '局 / 333源分片 | 最大轮数:', TARGET_EPOCHS)


## 代码2 / 6：检查GPU，加载notebook工具

只补缺少的W&B、tqdm、PyYAML，沿用Colab现有Torch。CUDA不可用会在加载大文件前停止。


In [ ]:
import importlib.util, subprocess, base64, hashlib
import torch

if not torch.version.cuda or not torch.cuda.is_available():
    raise RuntimeError('当前不是可用的GPU运行时。请在Colab“更改运行时类型”选择GPU，再从代码1运行；无需重装Torch。')
missing = [package for module, package in [
    ('wandb', 'wandb>=0.19,<1'), ('tqdm', 'tqdm>=4.66,<5'), ('yaml', 'PyYAML>=6,<7')
] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', *missing], check=True)
TOOLS = WORK / 'notebook_tools'
TOOLS.mkdir(exist_ok=True)
EMBEDDED_MODULES = {"notebook_state_bc.py":{"base64":"IiIiTm90ZWJvb2sgcHJvZ3Jlc3MgYW5kIFcmQiBtaXJyb3JpbmcgZm9yIHRoZSBleGlzdGluZyBzdGF0ZS1CQyBKU09OTCBsb2cuCgpUaGlzIG1vZHVsZSBuZXZlciBpbXBvcnRzIFRvcmNoLCBzdGFydHMgYSB0cmFpbmVyLCBvciBjcmVhdGVzIGEgVyZCIHJ1bi4gVGhlCm5vdGVib29rIG93bnMgdGhvc2Ugb3BlcmF0aW9ucy4gTG9nIGRlbGl2ZXJ5IGlzIGF0IGxlYXN0IG9uY2U6IHRoZSBkdXJhYmxlCmN1cnNvciBhZHZhbmNlcyBvbmx5IGFmdGVyIGBgcnVuLmxvZ2BgIGFjY2VwdHMgdGhlIGV2ZW50LiBBIGtlcm5lbCBjcmFzaCBpbiB0aGUKc21hbGwgaW50ZXJ2YWwgYmV0d2VlbiBkZWxpdmVyeSBhbmQgY3Vyc29yIHNhdmUgY2FuIHJlcGxheSB0aGF0IGZpbmFsIGV2ZW50LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgdGltZQppbXBvcnQgd2FybmluZ3MKCgpkZWYgX2ludGVnZXIodmFsdWUsIG5hbWUsIG1pbmltdW09MCk6CiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBib29sKSBvciBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgaW50KSBvciB2YWx1ZSA8IG1pbmltdW06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntuYW1lfSBtdXN0IGJlIGFuIGludGVnZXIgPj0ge21pbmltdW19IikKICAgIHJldHVybiB2YWx1ZQoKCmRlZiByZXN1bWVfY29tcGxldGVkKHByb2dyZXNzLCBzZWF0c19wZXJfZXBvY2gpOgogICAgIiIiQ2hlY2twb2ludCBlcG9jaHMgYXJlIHplcm8gYmFzZWQ7IG5leHRfZXBpc29kZSBjb3VudHMgY29tcGxldGVkIHNlYXRzLiIiIgogICAgc2VhdHMgPSBfaW50ZWdlcihzZWF0c19wZXJfZXBvY2gsICJzZWF0c19wZXJfZXBvY2giLCAxKQogICAgZXBvY2ggPSBfaW50ZWdlcihwcm9ncmVzcy5nZXQoImVwb2NoIiksICJlcG9jaCIpCiAgICBuZXh0X2VwaXNvZGUgPSBfaW50ZWdlcihwcm9ncmVzcy5nZXQoIm5leHRfZXBpc29kZSIpLCAibmV4dF9lcGlzb2RlIikKICAgIGlmIG5leHRfZXBpc29kZSA+IHNlYXRzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNoZWNrcG9pbnQgbmV4dF9lcGlzb2RlIGV4Y2VlZHMgdGhpcyBkYXRhc2V0J3MgZXBvY2giKQogICAgcmV0dXJuIGVwb2NoICogc2VhdHMgKyBuZXh0X2VwaXNvZGUKCgpkZWYgY29tcGxldGVkX3NlYXRzKHJlY29yZCwgc2VhdHNfcGVyX2Vwb2NoKToKICAgICIiIlRyYWluIHJvd3MgdXNlIG9uZS1iYXNlZCBlcG9jaCBhbmQgYSB6ZXJvLWJhc2VkIGNvbXBsZXRlZCBlcGlzb2RlIGluZGV4LiIiIgogICAgaWYgcmVjb3JkLmdldCgicGhhc2UiKSAhPSAic3RhdGVfYmNfdHJhaW4iOgogICAgICAgIHJldHVybiBOb25lICAjIEluaXRpYWwvbWlkLWVwb2NoIHZhbGlkYXRpb24gaXMgbm90IGFuIGVwb2NoIGNvbXBsZXRpb24uCiAgICBzZWF0cyA9IF9pbnRlZ2VyKHNlYXRzX3Blcl9lcG9jaCwgInNlYXRzX3Blcl9lcG9jaCIsIDEpCiAgICBlcG9jaCA9IF9pbnRlZ2VyKHJlY29yZC5nZXQoImVwb2NoIiksICJlcG9jaCIsIDEpCiAgICBpbmRleCA9IF9pbnRlZ2VyKHJlY29yZC5nZXQoImVwaXNvZGVfaW5kZXgiKSwgImVwaXNvZGVfaW5kZXgiKQogICAgaWYgaW5kZXggPj0gc2VhdHM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTG9nZ2VkIGVwaXNvZGUgaW5kZXggZXhjZWVkcyB0aGlzIGRhdGFzZXQncyBlcG9jaCIpCiAgICByZXR1cm4gKGVwb2NoIC0gMSkgKiBzZWF0cyArIGluZGV4ICsgMQoKCmRlZiBtZXRyaWNfcGF5bG9hZChyZWNvcmQpOgogICAgIiIiTWFwIGZpbml0ZSBzY2FsYXIgbWV0cmljczsgZG8gbm90IGNvbXBhcmUgVyZCJ3Mgb3duIHN0ZXAgdG8gbW9kZWwgc3RlcHMuIiIiCiAgICBwaGFzZSA9IHJlY29yZC5nZXQoInBoYXNlIikKICAgIHByZWZpeCA9IHsic3RhdGVfYmNfdHJhaW4iOiAidHJhaW4iLCAic3RhdGVfYmNfdmFsaWRhdGlvbiI6ICJ2YWxpZGF0aW9uIn0uZ2V0KHBoYXNlKQogICAgaWYgcHJlZml4IGlzIE5vbmU6CiAgICAgICAgaWYgcGhhc2UgIT0gInN0YXRlX2JjX3N0YXJ0IjoKICAgICAgICAgICAgcmV0dXJuIHt9CiAgICAgICAgcmVzdWx0ID0ge30KICAgICAgICBmb3Iga2V5IGluICgicGFyYW1ldGVyX2NvdW50IiwgInRyYWluYWJsZV9wYXJhbWV0ZXJfY291bnQiLCAicmFtX2xpbWl0X2dpYiIpOgogICAgICAgICAgICB2YWx1ZSA9IHJlY29yZC5nZXQoa2V5KQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAoaW50LCBmbG9hdCkpIGFuZCBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgYm9vbCkgYW5kIG1hdGguaXNmaW5pdGUodmFsdWUpOgogICAgICAgICAgICAgICAgcmVzdWx0W2YicnVuL3trZXl9Il0gPSB2YWx1ZQogICAgICAgIHJldHVybiByZXN1bHQKICAgIHN0ZXAgPSBfaW50ZWdlcihyZWNvcmQuZ2V0KCJvcHRpbWl6ZXJfc3RlcHMiKSwgIm9wdGltaXplcl9zdGVwcyIpCiAgICByZXN1bHQgPSB7InRyYWluZXIvb3B0aW1pemVyX3N0ZXBzIjogc3RlcH0KICAgIGZvciBrZXksIHZhbHVlIGluIHJlY29yZC5pdGVtcygpOgogICAgICAgIGlmIGtleSBpbiB7Im9wdGltaXplcl9zdGVwcyIsICJlbGFwc2VkX3NlY29uZHMifToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBib29sKToKICAgICAgICAgICAgcmVzdWx0W2Yie3ByZWZpeH0ve2tleX0iXSA9IGludCh2YWx1ZSkKICAgICAgICBlbGlmIGlzaW5zdGFuY2UodmFsdWUsIChpbnQsIGZsb2F0KSkgYW5kIG1hdGguaXNmaW5pdGUodmFsdWUpOgogICAgICAgICAgICByZXN1bHRbZiJ7cHJlZml4fS97a2V5fSJdID0gdmFsdWUKICAgIGlmIHBoYXNlPT0nc3RhdGVfYmNfdHJhaW4nIGFuZCByZWNvcmQuZ2V0KCdhZ2dyZWdhdGlvbicpPT0nb3B0aW1pemVyX3VwZGF0ZV93aW5kb3dfdjEnOgogICAgICAgIGZvciBrZXksdmFsdWUgaW4gbGlzdChyZXN1bHQuaXRlbXMoKSk6CiAgICAgICAgICAgIGlmIGtleS5zdGFydHN3aXRoKCd0cmFpbi8nKToKICAgICAgICAgICAgICAgIHJlc3VsdFsndHJhaW5fd2luZG93Lycra2V5W2xlbigndHJhaW4vJyk6XV09dmFsdWUKICAgIHJldHVybiByZXN1bHQKCgpkZWYgY29uZmlndXJlX3dhbmRiKHJ1bik6CiAgICAiIiJBIGN1c3RvbSB4IGF4aXMgcHJlc2VydmVzIHRyYWluIGFuZCB2YWxpZGF0aW9uIGV2ZW50cyBhdCB0aGUgc2FtZSBzdGVwLiIiIgogICAgcnVuLmRlZmluZV9tZXRyaWMoInRyYWluZXIvb3B0aW1pemVyX3N0ZXBzIikKICAgIHJ1bi5kZWZpbmVfbWV0cmljKCJ0cmFpbi8qIiwgc3RlcF9tZXRyaWM9InRyYWluZXIvb3B0aW1pemVyX3N0ZXBzIiwgc3RlcF9zeW5jPVRydWUpCiAgICBydW4uZGVmaW5lX21ldHJpYygidmFsaWRhdGlvbi8qIiwgc3RlcF9tZXRyaWM9InRyYWluZXIvb3B0aW1pemVyX3N0ZXBzIiwgc3RlcF9zeW5jPVRydWUpCiAgICBydW4uZGVmaW5lX21ldHJpYygndHJhaW5fd2luZG93LyonLHN0ZXBfbWV0cmljPSd0cmFpbmVyL29wdGltaXplcl9zdGVwcycsc3RlcF9zeW5jPVRydWUpCgoKY2xhc3MgSlNPTkxUYWlsZXI6CiAgICAiIiJSZWFkIGNvbXBsZXRlIGFwcGVuZGVkIGxpbmVzIGFuZCBwZXJzaXN0IGV4cGxpY2l0bHkgYWNrbm93bGVkZ2VkIG9mZnNldHMuCgogICAgQ3Vyc29yIGlkZW50aXR5IHNob3VsZCBiZSB0aGUgVyZCIHJ1biBJRC4gUHJlZml4IGFuZCBuZWFyLW9mZnNldCBhbmNob3JzCiAgICBhbGxvdyBpZGVudGljYWwgRHJpdmUtcmVzdG9yZWQgY29waWVzIHdoaWxlIGRldGVjdGluZyB0cnVuY2F0aW9uIG9yIGxvZwogICAgcmVwbGFjZW1lbnQuIElmIHRoZSBsb2cgcm9sbHMgYmFjaywgcmVwbGF5IGlzIGV4cGxpY2l0IHJhdGhlciB0aGFuIHNpbGVudGx5CiAgICBza2lwcGluZyBvbGQgYnl0ZXMuIEN1cnNvciBmaWxlcyBhcmUgc21hbGwgSlNPTiwgbmV2ZXIgbW9kZWwvZGF0YSB0ZW5zb3JzLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhdGgsIGN1cnNvcl9wYXRoPU5vbmUsICosIGlkZW50aXR5PSJkZWZhdWx0Iiwgc2tpcF9ub25fanNvbj1GYWxzZSk6CiAgICAgICAgc2VsZi5wYXRoID0gUGF0aChwYXRoKQogICAgICAgIHNlbGYuc2tpcF9ub25fanNvbiA9IGJvb2woc2tpcF9ub25fanNvbikKICAgICAgICBzZWxmLmN1cnNvcl9wYXRoID0gUGF0aChjdXJzb3JfcGF0aCkgaWYgY3Vyc29yX3BhdGggaXMgbm90IE5vbmUgZWxzZSBOb25lCiAgICAgICAgc2VsZi5pZGVudGl0eSA9IHN0cihpZGVudGl0eSkKICAgICAgICBzZWxmLm9mZnNldCA9IDAKICAgICAgICBzZWxmLnByZWZpeF9sZW5ndGggPSAwCiAgICAgICAgc2VsZi5wcmVmaXhfc2hhMjU2ID0gTm9uZQogICAgICAgIHNlbGYuYW5jaG9yX3N0YXJ0ID0gMAogICAgICAgIHNlbGYuYW5jaG9yX3NoYTI1NiA9IE5vbmUKICAgICAgICBzZWxmLnJlc2V0cyA9IDAKICAgICAgICBpZiBzZWxmLmN1cnNvcl9wYXRoIGlzIG5vdCBOb25lIGFuZCBzZWxmLmN1cnNvcl9wYXRoLmV4aXN0cygpOgogICAgICAgICAgICBzYXZlZCA9IGpzb24ubG9hZHMoc2VsZi5jdXJzb3JfcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgICAgIGlmIHNhdmVkLmdldCgiaWRlbnRpdHkiKSA9PSBzZWxmLmlkZW50aXR5OgogICAgICAgICAgICAgICAgc2VsZi5vZmZzZXQgPSBfaW50ZWdlcihzYXZlZC5nZXQoIm9mZnNldCIpLCAiY3Vyc29yIG9mZnNldCIpCiAgICAgICAgICAgICAgICBzZWxmLnByZWZpeF9sZW5ndGggPSBfaW50ZWdlcihzYXZlZC5nZXQoInByZWZpeF9sZW5ndGgiLCAwKSwgInByZWZpeF9sZW5ndGgiKQogICAgICAgICAgICAgICAgc2VsZi5wcmVmaXhfc2hhMjU2ID0gc2F2ZWQuZ2V0KCJwcmVmaXhfc2hhMjU2IikKICAgICAgICAgICAgICAgIHNlbGYuYW5jaG9yX3N0YXJ0ID0gX2ludGVnZXIoc2F2ZWQuZ2V0KCJhbmNob3Jfc3RhcnQiLCAwKSwgImFuY2hvcl9zdGFydCIpCiAgICAgICAgICAgICAgICBzZWxmLmFuY2hvcl9zaGEyNTYgPSBzYXZlZC5nZXQoImFuY2hvcl9zaGEyNTYiKQoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBfZGlnZXN0KHN0cmVhbSwgc3RhcnQsIGxlbmd0aCk6CiAgICAgICAgc3RyZWFtLnNlZWsoc3RhcnQpCiAgICAgICAgdmFsdWUgPSBzdHJlYW0ucmVhZChsZW5ndGgpCiAgICAgICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHZhbHVlKS5oZXhkaWdlc3QoKQoKICAgIGRlZiBfcmVzZXQoc2VsZik6CiAgICAgICAgc2VsZi5vZmZzZXQgPSAwCiAgICAgICAgc2VsZi5wcmVmaXhfbGVuZ3RoID0gMAogICAgICAgIHNlbGYucHJlZml4X3NoYTI1NiA9IE5vbmUKICAgICAgICBzZWxmLmFuY2hvcl9zdGFydCA9IDAKICAgICAgICBzZWxmLmFuY2hvcl9zaGEyNTYgPSBOb25lCiAgICAgICAgc2VsZi5yZXNldHMgKz0gMQogICAgICAgIHdhcm5pbmdzLndhcm4oIlRyYWluaW5nIGxvZyB3YXMgdHJ1bmNhdGVkL3JlcGxhY2VkOyByZXBsYXlpbmcgaXRzIGF2YWlsYWJsZSBldmVudHMiLCBSdW50aW1lV2FybmluZykKCiAgICBkZWYgcGVuZGluZyhzZWxmKToKICAgICAgICAiIiJZaWVsZCAocmVjb3JkLCBieXRlX29mZnNldCk7IGFja25vd2xlZGdlIGVhY2ggb25seSBhZnRlciBkZWxpdmVyeS4iIiIKICAgICAgICBpZiBub3Qgc2VsZi5wYXRoLmV4aXN0cygpOgogICAgICAgICAgICByZXR1cm4KICAgICAgICB3aXRoIHNlbGYucGF0aC5vcGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICAgICAgc2l6ZSA9IG9zLmZzdGF0KHN0cmVhbS5maWxlbm8oKSkuc3Rfc2l6ZQogICAgICAgICAgICBjaGFuZ2VkID0gc2l6ZSA8IHNlbGYub2Zmc2V0CiAgICAgICAgICAgIGlmIHNlbGYucHJlZml4X2xlbmd0aDoKICAgICAgICAgICAgICAgIGNoYW5nZWQgfD0gc2l6ZSA8IHNlbGYucHJlZml4X2xlbmd0aCBvciBzZWxmLl9kaWdlc3Qoc3RyZWFtLCAwLCBzZWxmLnByZWZpeF9sZW5ndGgpICE9IHNlbGYucHJlZml4X3NoYTI1NgogICAgICAgICAgICBpZiBzZWxmLm9mZnNldCBhbmQgc2VsZi5hbmNob3Jfc2hhMjU2OgogICAgICAgICAgICAgICAgY2hhbmdlZCB8PSBzZWxmLl9kaWdlc3Qoc3RyZWFtLCBzZWxmLmFuY2hvcl9zdGFydCwgc2VsZi5vZmZzZXQgLSBzZWxmLmFuY2hvcl9zdGFydCkgIT0gc2VsZi5hbmNob3Jfc2hhMjU2CiAgICAgICAgICAgIGlmIGNoYW5nZWQ6CiAgICAgICAgICAgICAgICBzZWxmLl9yZXNldCgpCiAgICAgICAgICAgIGlmIG5vdCBzZWxmLnByZWZpeF9sZW5ndGggYW5kIHNpemU6CiAgICAgICAgICAgICAgICBzZWxmLnByZWZpeF9sZW5ndGggPSBtaW4oc2l6ZSwgMTAyNCkKICAgICAgICAgICAgICAgIHNlbGYucHJlZml4X3NoYTI1NiA9IHNlbGYuX2RpZ2VzdChzdHJlYW0sIDAsIHNlbGYucHJlZml4X2xlbmd0aCkKICAgICAgICAgICAgc3RyZWFtLnNlZWsoc2VsZi5vZmZzZXQpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICBzdGFydCA9IHN0cmVhbS50ZWxsKCkKICAgICAgICAgICAgICAgIGxpbmUgPSBzdHJlYW0ucmVhZGxpbmUoOCAqIDEwMjQqKjIgKyAxKQogICAgICAgICAgICAgICAgaWYgbGVuKGxpbmUpID4gOCAqIDEwMjQqKjI6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlRyYWluaW5nIEpTT05MIGxpbmUgZXhjZWVkcyA4TWlCIGF0IGJ5dGUge3N0YXJ0fSIpCiAgICAgICAgICAgICAgICBpZiBub3QgbGluZSBvciBub3QgbGluZS5lbmRzd2l0aChiIlxuIik6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuICAjIEEgY29uY3VycmVudCB3cml0ZXIgY2FuIGZpbmlzaCB0aGlzIHJvdyBsYXRlci4KICAgICAgICAgICAgICAgIGVuZCA9IHN0cmVhbS50ZWxsKCkKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICByZWNvcmQgPSBqc29uLmxvYWRzKGxpbmUpCiAgICAgICAgICAgICAgICBleGNlcHQgKFZhbHVlRXJyb3IsIFVuaWNvZGVEZWNvZGVFcnJvcikgYXMgZXJyb3I6CiAgICAgICAgICAgICAgICAgICAgaWYgc2VsZi5za2lwX25vbl9qc29uOgogICAgICAgICAgICAgICAgICAgICAgICBzZWxmLmFja25vd2xlZGdlKGVuZCkKICAgICAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBjb21wbGV0ZSB0cmFpbmluZyBKU09OTCBsaW5lIGF0IGJ5dGUge3N0YXJ0fSIpIGZyb20gZXJyb3IKICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJlY29yZCwgZGljdCk6CiAgICAgICAgICAgICAgICAgICAgaWYgc2VsZi5za2lwX25vbl9qc29uOgogICAgICAgICAgICAgICAgICAgICAgICBzZWxmLmFja25vd2xlZGdlKGVuZCkKICAgICAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVHJhaW5pbmcgSlNPTkwgcmVjb3JkIGF0IGJ5dGUge3N0YXJ0fSBpcyBub3QgYW4gb2JqZWN0IikKICAgICAgICAgICAgICAgIHlpZWxkIHJlY29yZCwgZW5kCgogICAgZGVmIGFja25vd2xlZGdlKHNlbGYsIG9mZnNldCk6CiAgICAgICAgb2Zmc2V0ID0gX2ludGVnZXIob2Zmc2V0LCAiYWNrbm93bGVkZ2VkIG9mZnNldCIpCiAgICAgICAgaWYgb2Zmc2V0IDwgc2VsZi5vZmZzZXQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNhbm5vdCBhY2tub3dsZWRnZSBhIHByZXZpb3VzIGxvZyBwb3NpdGlvbiIpCiAgICAgICAgd2l0aCBzZWxmLnBhdGgub3BlbigicmIiKSBhcyBzdHJlYW06CiAgICAgICAgICAgIGlmIG9zLmZzdGF0KHN0cmVhbS5maWxlbm8oKSkuc3Rfc2l6ZSA8IG9mZnNldDoKICAgICAgICAgICAgICAgIHJhaXNlIE9TRXJyb3IoIlRyYWluaW5nIGxvZyB0cnVuY2F0ZWQgYmVmb3JlIGFja25vd2xlZGdlbWVudCIpCiAgICAgICAgICAgIHNlbGYuYW5jaG9yX3N0YXJ0ID0gbWF4KDAsIG9mZnNldCAtIDEyOCkKICAgICAgICAgICAgc2VsZi5hbmNob3Jfc2hhMjU2ID0gc2VsZi5fZGlnZXN0KHN0cmVhbSwgc2VsZi5hbmNob3Jfc3RhcnQsIG9mZnNldCAtIHNlbGYuYW5jaG9yX3N0YXJ0KQogICAgICAgIHNlbGYub2Zmc2V0ID0gb2Zmc2V0CiAgICAgICAgaWYgc2VsZi5jdXJzb3JfcGF0aCBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5jdXJzb3JfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICB0ZW1wb3JhcnkgPSBzZWxmLmN1cnNvcl9wYXRoLndpdGhfbmFtZShzZWxmLmN1cnNvcl9wYXRoLm5hbWUgKyAiLnRtcCIpCiAgICAgICAgICAgIHRlbXBvcmFyeS53cml0ZV90ZXh0KGpzb24uZHVtcHMoewogICAgICAgICAgICAgICAgImlkZW50aXR5Ijogc2VsZi5pZGVudGl0eSwgIm9mZnNldCI6IHNlbGYub2Zmc2V0LAogICAgICAgICAgICAgICAgInByZWZpeF9sZW5ndGgiOiBzZWxmLnByZWZpeF9sZW5ndGgsICJwcmVmaXhfc2hhMjU2Ijogc2VsZi5wcmVmaXhfc2hhMjU2LAogICAgICAgICAgICAgICAgImFuY2hvcl9zdGFydCI6IHNlbGYuYW5jaG9yX3N0YXJ0LCAiYW5jaG9yX3NoYTI1NiI6IHNlbGYuYW5jaG9yX3NoYTI1NiwKICAgICAgICAgICAgfSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAgIHRlbXBvcmFyeS5yZXBsYWNlKHNlbGYuY3Vyc29yX3BhdGgpCgoKZGVmIGxvZ190YWlsKHBhdGgsIGxpbmVzPTYwLCBtYXhfYnl0ZXM9NjU1MzYpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOgogICAgICAgIHJldHVybiAiKHRyYWluZXIgbG9nIGRvZXMgbm90IGV4aXN0KSIKICAgIHdpdGggcGF0aC5vcGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICBzdHJlYW0uc2VlayhtYXgoMCwgcGF0aC5zdGF0KCkuc3Rfc2l6ZSAtIG1heF9ieXRlcykpCiAgICAgICAgcmV0dXJuICJcbiIuam9pbihzdHJlYW0ucmVhZCgpLmRlY29kZSgidXRmLTgiLCBlcnJvcnM9InJlcGxhY2UiKS5zcGxpdGxpbmVzKClbLWxpbmVzOl0pCgoKZGVmIG1vbml0b3JfcHJvY2Vzcyhwcm9jZXNzLCBtZXRyaWNzX3BhdGgsIHRyYWluX2xvZ19wYXRoLCAqLCBydW49Tm9uZSwgdGFyZ2V0X2Vwb2Nocz0zLAogICAgICAgICAgICAgICAgICAgIHNlYXRzX3Blcl9lcG9jaD0zNTAsIGluaXRpYWxfY29tcGxldGVkPTAsIGN1cnNvcl9wYXRoPU5vbmUsCiAgICAgICAgICAgICAgICAgICAgcG9sbF9zZWNvbmRzPTIsIGJhcj1Ob25lLCBvbl9wb2xsPU5vbmUpOgogICAgIiIiTWlycm9yIGxvZ3MgYW5kIG1vbml0b3IgYW4gZXhpc3Rpbmcgc3VicHJvY2VzcyB3aXRob3V0IGFsdGVyaW5nIHRyYWluaW5nLgoKICAgIEtleWJvYXJkSW50ZXJydXB0IGNsb3NlcyB0aGUgZGlzcGxheSBhbmQgcHJvcGFnYXRlcy4gVGhlIG5vdGVib29rIG11c3QKICAgIGV4cGxpY2l0bHkgZGVjaWRlIHdoZXRoZXIgaXRzIG93bmVkIHRyYWluZXIgc2hvdWxkIHN0b3Agb3IgY29udGludWUuCiAgICBUZW1wb3JhcnkgVyZCIGRlbGl2ZXJ5IGZhaWx1cmVzIGxlYXZlIHRoZWlyIGN1cnNvciBwZW5kaW5nIGFuZCBhcmUgcmV0cmllZAogICAgd2hpbGUgdHJhaW5pbmcgY29udGludWVzLiBPbiBzdWNjZXNzZnVsIHRyYWluZXIgZXhpdCwgdW5yZXNvbHZlZCBkZWxpdmVyeQogICAgcmV0dXJucyBgYGxvZ2dpbmdfc3RhdHVzPSdwZW5kaW5nJ2BgIGFuZCBwcmludHMgYW4gZXhwbGljaXQgd2FybmluZy4KICAgIGBgcnVuLmZpbmlzaGBgIGlzIGFsc28gdGhlIG5vdGVib29rJ3MgcmVzcG9uc2liaWxpdHkuCiAgICAiIiIKICAgIHRvdGFsID0gX2ludGVnZXIodGFyZ2V0X2Vwb2NocywgInRhcmdldF9lcG9jaHMiLCAxKSAqIF9pbnRlZ2VyKHNlYXRzX3Blcl9lcG9jaCwgInNlYXRzX3Blcl9lcG9jaCIsIDEpCiAgICBpbml0aWFsID0gX2ludGVnZXIoaW5pdGlhbF9jb21wbGV0ZWQsICJpbml0aWFsX2NvbXBsZXRlZCIpCiAgICBpZiBpbml0aWFsID4gdG90YWw6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ2hlY2twb2ludCBwcm9ncmVzcyBleGNlZWRzIHJlcXVlc3RlZCB0YXJnZXQgZXBvY2hzIikKICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHBvbGxfc2Vjb25kcykgb3IgcG9sbF9zZWNvbmRzIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicG9sbF9zZWNvbmRzIG11c3QgYmUgcG9zaXRpdmUiKQogICAgaWYgYmFyIGlzIE5vbmU6CiAgICAgICAgZnJvbSB0cWRtLmF1dG8gaW1wb3J0IHRxZG0KICAgICAgICBiYXIgPSB0cWRtKHRvdGFsPXRvdGFsLCBpbml0aWFsPWluaXRpYWwsIGRlc2M9IkJDIHNlYXRzIiwgdW5pdD0ic2VhdCIpCiAgICBpZGVudGl0eSA9IGdldGF0dHIocnVuLCAiaWQiLCAibm8td2FuZGIiKQogICAgdGFpbGVyID0gSlNPTkxUYWlsZXIobWV0cmljc19wYXRoLCBjdXJzb3JfcGF0aCwgaWRlbnRpdHk9aWRlbnRpdHkpCiAgICBkaXNwbGF5X3RhaWxlciA9IEpTT05MVGFpbGVyKG1ldHJpY3NfcGF0aCwgaWRlbnRpdHk9Im5vdGVib29rLWRpc3BsYXkiKQogICAgbGF0ZXN0ID0ge30KICAgIHZhbGlkYXRpb25zID0gW10KICAgIGNvbXBsZXRlZCA9IGluaXRpYWwKICAgIGxvZ2dpbmdfZXJyb3IgPSBOb25lCiAgICBtb25pdG9yX3N5bmNfZXJyb3JzID0gMAoKICAgIGRlZiBzeW5jX21vbml0b3JfbWV0YWRhdGEoKToKICAgICAgICBub25sb2NhbCBtb25pdG9yX3N5bmNfZXJyb3JzCiAgICAgICAgaWYgb25fcG9sbCBpcyBub3QgTm9uZToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgb25fcG9sbCgpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXJyb3I6CiAgICAgICAgICAgICAgICBtb25pdG9yX3N5bmNfZXJyb3JzICs9IDEKICAgICAgICAgICAgICAgIHByaW50KGYi55uR5o6n5YWD5pWw5o2u5ZCM5q2l5aSx6LSl77yIe3R5cGUoZXJyb3IpLl9fbmFtZV9ffe+8ie+8m+iuree7g+e7p+e7re+8jOacrOasoeWQjOatpeacquaIkOWKn+OAgiIpCgogICAgY2xhc3MgRGVsaXZlcnlGYWlsdXJlKEV4Y2VwdGlvbik6CiAgICAgICAgZGVmIF9faW5pdF9fKHNlbGYsIG9yaWdpbmFsKToKICAgICAgICAgICAgc2VsZi5vcmlnaW5hbCA9IG9yaWdpbmFsCgogICAgZGVmIGNvbnN1bWVfZGlzcGxheSgpOgogICAgICAgIG5vbmxvY2FsIGNvbXBsZXRlZAogICAgICAgIHBvc3RmaXggPSBOb25lCiAgICAgICAgZm9yIHJlY29yZCwgb2Zmc2V0IGluIGRpc3BsYXlfdGFpbGVyLnBlbmRpbmcoKToKICAgICAgICAgICAgcGhhc2UgPSByZWNvcmQuZ2V0KCJwaGFzZSIpCiAgICAgICAgICAgIGlmIHBoYXNlIGluICgic3RhdGVfYmNfdHJhaW4iLCAic3RhdGVfYmNfdmFsaWRhdGlvbiIpOgogICAgICAgICAgICAgICAgbGF0ZXN0W3BoYXNlXSA9IHJlY29yZAogICAgICAgICAgICAgICAgaWYgcGhhc2UgPT0gInN0YXRlX2JjX3ZhbGlkYXRpb24iOgogICAgICAgICAgICAgICAgICAgIHZhbGlkYXRpb25zLmFwcGVuZChyZWNvcmQpCiAgICAgICAgICAgICAgICBjb3VudCA9IGNvbXBsZXRlZF9zZWF0cyhyZWNvcmQsIHNlYXRzX3Blcl9lcG9jaCkKICAgICAgICAgICAgICAgIGlmIGNvdW50IGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIGlmIGNvdW50ID4gdG90YWw6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRyYWluaW5nIGxvZyBjb250YWlucyBlcG9jaHMgYmV5b25kIHRoZSBub3RlYm9vayB0YXJnZXQiKQogICAgICAgICAgICAgICAgICAgIGNvbXBsZXRlZCA9IG1heChjb21wbGV0ZWQsIGNvdW50KQogICAgICAgICAgICAgICAgICAgIGlmIGNvbXBsZXRlZCA+IGJhci5uOgogICAgICAgICAgICAgICAgICAgICAgICBiYXIudXBkYXRlKGNvbXBsZXRlZCAtIGJhci5uKQogICAgICAgICAgICAgICAgcG9zdGZpeCA9IHsicGhhc2UiOiAidmFsaWRhdGlvbiIgaWYgcGhhc2UuZW5kc3dpdGgoInZhbGlkYXRpb24iKSBlbHNlICJ0cmFpbmluZyIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJlcG9jaCI6IHJlY29yZC5nZXQoImVwb2NoIiksICJzdGVwIjogcmVjb3JkLmdldCgib3B0aW1pemVyX3N0ZXBzIil9CiAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHJlY29yZC5nZXQoImxvc3MiKSwgKGZsb2F0LCBpbnQpKToKICAgICAgICAgICAgICAgICAgICBwb3N0Zml4WyJsb3NzIl0gPSBmIntyZWNvcmRbJ2xvc3MnXTouNGZ9IgogICAgICAgICAgICAgICAgaWYgImFtcF9zY2FsZSIgaW4gcmVjb3JkOgogICAgICAgICAgICAgICAgICAgIHBvc3RmaXhbInNjYWxlIl0gPSByZWNvcmRbImFtcF9zY2FsZSJdCiAgICAgICAgICAgIGRpc3BsYXlfdGFpbGVyLmFja25vd2xlZGdlKG9mZnNldCkKICAgICAgICBpZiBwb3N0Zml4IGlzIG5vdCBOb25lOgogICAgICAgICAgICBiYXIuc2V0X3Bvc3RmaXgocG9zdGZpeCkKCiAgICBkZWYgY29uc3VtZSgpOgogICAgICAgIGZvciByZWNvcmQsIG9mZnNldCBpbiB0YWlsZXIucGVuZGluZygpOgogICAgICAgICAgICBwYXlsb2FkID0gbWV0cmljX3BheWxvYWQocmVjb3JkKQogICAgICAgICAgICBpZiBydW4gaXMgbm90IE5vbmUgYW5kIHBheWxvYWQ6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcnVuLmxvZyhwYXlsb2FkKSAgIyBEbyBub3Qgc3VwcGx5IHN0ZXA9b3B0aW1pemVyX3N0ZXBzLgogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlcnJvcjoKICAgICAgICAgICAgICAgICAgICByYWlzZSBEZWxpdmVyeUZhaWx1cmUoZXJyb3IpIGZyb20gZXJyb3IKICAgICAgICAgICAgdGFpbGVyLmFja25vd2xlZGdlKG9mZnNldCkKCiAgICB0cnk6CiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgY29uc3VtZV9kaXNwbGF5KCkgICMgSW5kZXBlbmRlbnQgb2YgVyZCIGNvbm5lY3Rpdml0eS9jdXJzb3Igc3RhdGUuCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGNvbnN1bWUoKQogICAgICAgICAgICAgICAgbG9nZ2luZ19lcnJvciA9IE5vbmUKICAgICAgICAgICAgZXhjZXB0IERlbGl2ZXJ5RmFpbHVyZSBhcyBlcnJvcjoKICAgICAgICAgICAgICAgIGlmIGxvZ2dpbmdfZXJyb3IgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICBwcmludCgiVyZC5pel5b+X5pqC5pyq6YCB6L6+77yM6K6t57uD57un57ut77yM5b6F5Y+R6YCB6K6w5b2V5bCG6Ieq5Yqo6YeN6K+V77yb6K+35Yu/6YeN5ZCvdHJhaW5lcuOAgiIpCiAgICAgICAgICAgICAgICBsb2dnaW5nX2Vycm9yID0gdHlwZShlcnJvci5vcmlnaW5hbCkuX19uYW1lX18KICAgICAgICAgICAgc3luY19tb25pdG9yX21ldGFkYXRhKCkKICAgICAgICAgICAgY29kZSA9IHByb2Nlc3MucG9sbCgpCiAgICAgICAgICAgIGlmIGNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBjb25zdW1lX2Rpc3BsYXkoKQogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIGNvbnN1bWUoKSAgIyBEcmFpbiBmaW5hbCByZWNvcmRzIGFmdGVyIHRoZSBjaGlsZCBoYXMgZmx1c2hlZC9leGl0ZWQuCiAgICAgICAgICAgICAgICAgICAgbG9nZ2luZ19lcnJvciA9IE5vbmUKICAgICAgICAgICAgICAgIGV4Y2VwdCBEZWxpdmVyeUZhaWx1cmUgYXMgZXJyb3I6CiAgICAgICAgICAgICAgICAgICAgbG9nZ2luZ19lcnJvciA9IHR5cGUoZXJyb3Iub3JpZ2luYWwpLl9fbmFtZV9fCiAgICAgICAgICAgICAgICBzeW5jX21vbml0b3JfbWV0YWRhdGEoKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgdGltZS5zbGVlcChwb2xsX3NlY29uZHMpCiAgICBleGNlcHQgS2V5Ym9hcmRJbnRlcnJ1cHQ6CiAgICAgICAgcHJpbnQoIuebkeaOp+W3suaaguWBnO+8m3RyYWluZXLmnKrooqvmnKznm5HmjqflgZzmraLjgILph43mlrDov5DooYznm5HmjqdjZWxs5Y+v57un57ut5p+l55yL5ZCM5LiA6L+b56iL44CCIikKICAgICAgICByYWlzZQogICAgZmluYWxseToKICAgICAgICBiYXIuY2xvc2UoKQogICAgaWYgY29kZToKICAgICAgICBmYWlsdXJlX3RhaWwgPSBsb2dfdGFpbCh0cmFpbl9sb2dfcGF0aCkKICAgICAgICBwcmludChmInRyYWluZXIgcmV0dXJuY29kZToge2NvZGV9XG4tLS0gdHJhaW5pbmcubG9nIHRhaWwgLS0tXG57ZmFpbHVyZV90YWlsfSIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiQkMgdHJhaW5lciBleGl0ZWQgd2l0aCBjb2RlIHtjb2RlfS5cblxue2ZhaWx1cmVfdGFpbH0iKQogICAgaWYgbG9nZ2luZ19lcnJvcjoKICAgICAgICBwcmludCgidHJhaW5lcuW3suWujOaIkO+8jOS9hlcmQuS7jeacieW+heWPkemAgeaXpeW/l++8m0pTT05M5Y+KY3Vyc29y5bey5L+d55WZ77yM5Y+v6YeN5paw6L+Q6KGM55uR5o6n6KGl5Lyg44CCIikKICAgIHJldHVybiB7InJldHVybmNvZGUiOiBjb2RlLCAiY29tcGxldGVkX3NlYXRzIjogY29tcGxldGVkLCAidGFyZ2V0X3NlYXRzIjogdG90YWwsCiAgICAgICAgICAgICJsYXRlc3QiOiBsYXRlc3QsICJ2YWxpZGF0aW9uX2V2ZW50cyI6IGxlbih2YWxpZGF0aW9ucyksICJsb2dfcmVzZXRzIjogdGFpbGVyLnJlc2V0cywKICAgICAgICAgICAgImxvZ2dpbmdfc3RhdHVzIjogInBlbmRpbmciIGlmIGxvZ2dpbmdfZXJyb3IgZWxzZSAiYWNjZXB0ZWQiLAogICAgICAgICAgICAibG9nZ2luZ19lcnJvcl90eXBlIjogbG9nZ2luZ19lcnJvciwKICAgICAgICAgICAgIm1vbml0b3Jfc3luY19lcnJvcnMiOiBtb25pdG9yX3N5bmNfZXJyb3JzfQo=","sha256":"99c2406ccad26faa3a256eaa04ff515a93c4ab4f094c1469c83d82c0a63a18d6"},"notebook_state_bc_setup.py":{"base64":"...","sha256":"1254aeb4f8efc6a319b4810ad78a6727ebbbe938876a1b0fbd0d8169ea51abfa"},"notebook_state_bc_launch.py":{"base64":"IiIiT25lIGR1cmFibGUgQkMgdHJhaW5lciBwZXIgbm90ZWJvb2sgd29yayBkaXJlY3Rvcnk7IG5vIG1vZGVsIGltcG9ydHMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHRyYWNlYmFjawoKCmRlZiBhdG9taWNfanNvbihwYXRoLCB2YWx1ZSk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdGVtcG9yYXJ5ID0gcGF0aC53aXRoX25hbWUocGF0aC5uYW1lICsgJy50bXAnKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSwgaW5kZW50PTIpLCBlbmNvZGluZz0ndXRmLTgnKQogICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKCgpkZWYgcHJvY2Vzc19pZGVudGl0eShwaWQpOgogICAgIiIiTGludXggUElEIHBsdXMgc3RhcnQgdGlja3MgcHJldmVudHMgYXR0YWNoaW5nIHRvIGEgcmV1c2VkIFBJRC4iIiIKICAgIHRyeToKICAgICAgICBmaWVsZHMgPSBQYXRoKGYnL3Byb2Mve2ludChwaWQpfS9zdGF0JykucmVhZF90ZXh0KCkucnNwbGl0KCcpJywgMSlbMV0uc3BsaXQoKQogICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvciwgUHJvY2Vzc0xvb2t1cEVycm9yLCBUeXBlRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZiBmaWVsZHNbMF0gaW4gKCdaJywgJ1gnKToKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIGZpZWxkc1sxOV0KCgpkZWYgYWxpdmUocmVjb3JkLCByb2xlKToKICAgIHBpZCwgc3RhcnQgPSByZWNvcmQuZ2V0KHJvbGUgKyAnX3BpZCcpLCByZWNvcmQuZ2V0KHJvbGUgKyAnX3N0YXJ0JykKICAgIHJldHVybiBib29sKHBpZCBhbmQgc3RhcnQgYW5kIHByb2Nlc3NfaWRlbnRpdHkocGlkKSA9PSBzdGFydCkKCgpkZWYgcmVhZF9zdGF0ZShwYXRoKToKICAgIHRyeToKICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhQYXRoKHBhdGgpLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnKSkKICAgIGV4Y2VwdCBGaWxlTm90Rm91bmRFcnJvcjoKICAgICAgICByZXR1cm4ge30KCgpjbGFzcyBUcmFpbmluZ0pvYjoKICAgICIiIlBvcGVuLWNvbXBhdGlibGUgcG9sbCBmb3IgYm90aCBvcmlnaW5hbCBhbmQgcmVhdHRhY2hlZCBub3RlYm9vayBrZXJuZWxzLiIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHN0YXRlX3BhdGgpOgogICAgICAgIHNlbGYuc3RhdGVfcGF0aCA9IFBhdGgoc3RhdGVfcGF0aCkKCiAgICBAcHJvcGVydHkKICAgIGRlZiBwaWQoc2VsZik6CiAgICAgICAgcmVjb3JkID0gcmVhZF9zdGF0ZShzZWxmLnN0YXRlX3BhdGgpCiAgICAgICAgcmV0dXJuIHJlY29yZC5nZXQoJ3RyYWluZXJfcGlkJykgb3IgcmVjb3JkLmdldCgnd29ya2VyX3BpZCcpCgogICAgQHByb3BlcnR5CiAgICBkZWYgcmV0dXJuY29kZShzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5wb2xsKCkKCiAgICBkZWYgcG9sbChzZWxmKToKICAgICAgICByZWNvcmQgPSByZWFkX3N0YXRlKHNlbGYuc3RhdGVfcGF0aCkKICAgICAgICBpZiBhbGl2ZShyZWNvcmQsICd3b3JrZXInKSBvciBhbGl2ZShyZWNvcmQsICd0cmFpbmVyJyk6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgIyBOZXZlciBtaXN0YWtlIGEgZGlzYXBwZWFyZWQgc3VwZXJ2aXNvciAvIHJlY2xhaW1lZCBydW50aW1lIGZvciBzdWNjZXNzLgogICAgICAgIGlmIHJlY29yZC5nZXQoJ3N0YXRlJykgPT0gJ2V4aXRlZCcgYW5kIGlzaW5zdGFuY2UocmVjb3JkLmdldCgncmV0dXJuY29kZScpLCBpbnQpOgogICAgICAgICAgICByZXR1cm4gcmVjb3JkWydyZXR1cm5jb2RlJ10KICAgICAgICByZXR1cm4gMTI1CgoKZGVmIGxhdW5jaF9vcl9hdHRhY2goYXJndiwgKiwgc291cmNlX3Jvb3QsIHdvcmtzcGFjZSwgbG9nX3BhdGgsIGVudl9vdmVycmlkZXM9Tm9uZSwKICAgICAgICAgICAgICAgICAgICAgcmVzdGFydF9jb21wbGV0ZWQ9RmFsc2UpOgogICAgIiIiQXR0YWNoIHRvIGxpdmUgam9iczsgb3B0aW9uYWxseSByZXJ1biBhIG5vcm1hbGx5IGV4aXRlZCBpZGVudGljYWwgY29tbWFuZC4KCiAgICBBIGNvbnRyb2xsZXIgdGhhdCBleGl0cyBzdWNjZXNzZnVsbHkgd2hpbGUgd2FpdGluZyBmb3IgbW9yZSBkYXRhIGNhbiBiZQogICAgcmVzdGFydGVkIGV4cGxpY2l0bHkuIFRoZSBkZWZhdWx0IHJldGFpbnMgY29tcGxldGVkLWNvbW1hbmQgaWRlbXBvdGVuY3ksCiAgICBhbmQgZXZlbiBhbiBleHBsaWNpdCByZXN0YXJ0IG5ldmVyIGR1cGxpY2F0ZXMgYSBzdGlsbC1saXZlIHByb2Nlc3MuCiAgICAiIiIKICAgIGlmIHN5cy5wbGF0Zm9ybSAhPSAnbGludXgnOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignVGhpcyBDb2xhYiBwcm9jZXNzIGxhdW5jaGVyIHJlcXVpcmVzIExpbnV4JykKICAgIGltcG9ydCBmY250bAoKICAgIHNvdXJjZV9yb290LCB3b3Jrc3BhY2UgPSBQYXRoKHNvdXJjZV9yb290KS5yZXNvbHZlKCksIFBhdGgod29ya3NwYWNlKS5yZXNvbHZlKCkKICAgIHdvcmtzcGFjZS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb250cm9sID0gd29ya3NwYWNlIC8gJy5ub3RlYm9vaycKICAgIGNvbnRyb2wubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIHN0YXRlX3BhdGggPSBjb250cm9sIC8gJ3RyYWluZXJfcHJvY2Vzcy5qc29uJwogICAgYXJndiA9IFtzdHIoYXJndW1lbnQpIGZvciBhcmd1bWVudCBpbiBhcmd2XQogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoanNvbi5kdW1wcyhhcmd2LCBzZXBhcmF0b3JzPSgnLCcsICc6JykpLmVuY29kZSgpKS5oZXhkaWdlc3QoKQogICAgd2l0aCAoY29udHJvbCAvICdsYXVuY2gubG9jaycpLm9wZW4oJ2EnKSBhcyBsb2NrOgogICAgICAgIGZjbnRsLmZsb2NrKGxvY2ssIGZjbnRsLkxPQ0tfRVgpCiAgICAgICAgcmVjb3JkID0gcmVhZF9zdGF0ZShzdGF0ZV9wYXRoKQogICAgICAgIGFjdGl2ZSA9IGFsaXZlKHJlY29yZCwgJ3dvcmtlcicpIG9yIGFsaXZlKHJlY29yZCwgJ3RyYWluZXInKQogICAgICAgIGlmIGFjdGl2ZToKICAgICAgICAgICAgaWYgcmVjb3JkLmdldCgnYXJndl9zaGEyNTYnKSAhPSBkaWdlc3Q6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0EgdHJhaW5lciBpcyBhbHJlYWR5IHJ1bm5pbmcgd2l0aCBkaWZmZXJlbnQgYXJndW1lbnRzOyBhdHRhY2ggZmlyc3QnKQogICAgICAgICAgICBwcmludChmJ0F0dGFjaGVkIHRvIGV4aXN0aW5nIHRyYWluZXIgUElEIHtyZWNvcmQuZ2V0KCJ0cmFpbmVyX3BpZCIpfScsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIHJldHVybiBUcmFpbmluZ0pvYihzdGF0ZV9wYXRoKQogICAgICAgIGlmIChub3QgcmVzdGFydF9jb21wbGV0ZWQgYW5kIHJlY29yZC5nZXQoJ3N0YXRlJykgPT0gJ2V4aXRlZCcKICAgICAgICAgICAgICAgIGFuZCByZWNvcmQuZ2V0KCdyZXR1cm5jb2RlJykgPT0gMCBhbmQgcmVjb3JkLmdldCgnYXJndl9zaGEyNTYnKSA9PSBkaWdlc3QpOgogICAgICAgICAgICBwcmludCgnVGhpcyB0cmFpbmluZyBjb21tYW5kIGFscmVhZHkgZmluaXNoZWQgc3VjY2Vzc2Z1bGx5JywgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgcmV0dXJuIFRyYWluaW5nSm9iKHN0YXRlX3BhdGgpCiAgICAgICAgc3BlYyA9IHsKICAgICAgICAgICAgJ2FyZ3YnOiBhcmd2LCAnc291cmNlX3Jvb3QnOiBzdHIoc291cmNlX3Jvb3QpLCAnd29ya3NwYWNlJzogc3RyKHdvcmtzcGFjZSksCiAgICAgICAgICAgICdsb2dfcGF0aCc6IHN0cihQYXRoKGxvZ19wYXRoKS5yZXNvbHZlKCkpLCAnc3RhdGVfcGF0aCc6IHN0cihzdGF0ZV9wYXRoKSwKICAgICAgICAgICAgJ2FyZ3Zfc2hhMjU2JzogZGlnZXN0LAogICAgICAgICAgICAnZW52X292ZXJyaWRlcyc6IGVudl9vdmVycmlkZXMgb3Ige30sCiAgICAgICAgfQogICAgICAgIHNwZWNfcGF0aCA9IGNvbnRyb2wgLyAnbGF1bmNoX3NwZWMuanNvbicKICAgICAgICBhdG9taWNfanNvbihzcGVjX3BhdGgsIHNwZWMpCiAgICAgICAgd2l0aCAoY29udHJvbCAvICdzdXBlcnZpc29yLmxvZycpLm9wZW4oJ2EnLCBidWZmZXJpbmc9MSkgYXMgb3V0cHV0OgogICAgICAgICAgICB3b3JrZXIgPSBzdWJwcm9jZXNzLlBvcGVuKAogICAgICAgICAgICAgICAgW3N5cy5leGVjdXRhYmxlLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpKSwgJy0td29ya2VyJywgc3RyKHNwZWNfcGF0aCldLAogICAgICAgICAgICAgICAgY3dkPScvY29udGVudCcgaWYgUGF0aCgnL2NvbnRlbnQnKS5pc19kaXIoKSBlbHNlIHN0cih3b3Jrc3BhY2UpLAogICAgICAgICAgICAgICAgc3Rkb3V0PW91dHB1dCwgc3RkZXJyPXN1YnByb2Nlc3MuU1RET1VULCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICApCiAgICAgICAgIyBUaGUgd29ya2VyIHB1Ymxpc2hlcyBpdHMgb3duIGlkZW50aXR5IGJlZm9yZSBpbXBvcnRpbmcgLyBsYXVuY2hpbmcgVG9yY2guCiAgICAgICAgZGVhZGxpbmUgPSB0aW1lLm1vbm90b25pYygpICsgMTUKICAgICAgICB3aGlsZSB0aW1lLm1vbm90b25pYygpIDwgZGVhZGxpbmU6CiAgICAgICAgICAgIHJlY29yZCA9IHJlYWRfc3RhdGUoc3RhdGVfcGF0aCkKICAgICAgICAgICAgaWYgcmVjb3JkLmdldCgnd29ya2VyX3BpZCcpID09IHdvcmtlci5waWQ6CiAgICAgICAgICAgICAgICBwcmludChmJ1RyYWluZXIgc3VwZXJ2aXNvciBQSUQge3dvcmtlci5waWR9JywgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgICAgIHJldHVybiBUcmFpbmluZ0pvYihzdGF0ZV9wYXRoKQogICAgICAgICAgICBpZiB3b3JrZXIucG9sbCgpIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdUcmFpbmVyIHN1cGVydmlzb3IgZmFpbGVkOyBpbnNwZWN0ICcgKyBzdHIoY29udHJvbCAvICdzdXBlcnZpc29yLmxvZycpKQogICAgICAgICAgICB0aW1lLnNsZWVwKC4xKQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignVHJhaW5lciBpcyBzdGlsbCBzdGFydGluZzsgcmVydW4gdGhpcyBjZWxsIHRvIGF0dGFjaCwgZG8gbm90IHN0YXJ0IGFub3RoZXIgY29tbWFuZCcpCgoKZGVmIHdvcmtlcl9tYWluKHNwZWNfcGF0aCk6CiAgICBpbXBvcnQgZmNudGwKCiAgICBzcGVjID0ganNvbi5sb2FkcyhQYXRoKHNwZWNfcGF0aCkucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpKQogICAgd29ya3NwYWNlID0gUGF0aChzcGVjWyd3b3Jrc3BhY2UnXSkKICAgIHN0YXRlX3BhdGggPSBQYXRoKHNwZWNbJ3N0YXRlX3BhdGgnXSkKICAgIHJlY29yZCA9IHsKICAgICAgICAnYXJndl9zaGEyNTYnOiBzcGVjWydhcmd2X3NoYTI1NiddLCAnd29ya2VyX3BpZCc6IG9zLmdldHBpZCgpLAogICAgICAgICd3b3JrZXJfc3RhcnQnOiBwcm9jZXNzX2lkZW50aXR5KG9zLmdldHBpZCgpKSwgJ3N0YXRlJzogJ3N0YXJ0aW5nJywKICAgICAgICAnc3RhcnRlZF91bml4JzogdGltZS50aW1lKCksICdyZXR1cm5jb2RlJzogTm9uZSwKICAgIH0KICAgIGxvY2tfcGF0aCA9IHdvcmtzcGFjZSAvICcubm90ZWJvb2svdHJhaW5lci5sb2NrJwogICAgd2l0aCBsb2NrX3BhdGgub3BlbignYScpIGFzIGxvY2s6CiAgICAgICAgZmNudGwuZmxvY2sobG9jaywgZmNudGwuTE9DS19FWCB8IGZjbnRsLkxPQ0tfTkIpCiAgICAgICAgYXRvbWljX2pzb24oc3RhdGVfcGF0aCwgcmVjb3JkKQogICAgICAgIGNoaWxkID0gTm9uZQogICAgICAgIHRyeToKICAgICAgICAgICAgbG9nX3BhdGggPSBQYXRoKHNwZWNbJ2xvZ19wYXRoJ10pCiAgICAgICAgICAgIGxvZ19wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIGVudmlyb25tZW50ID0geyoqb3MuZW52aXJvbiwgKipzcGVjWydlbnZfb3ZlcnJpZGVzJ10sICdQWVRIT05VTkJVRkZFUkVEJzogJzEnfQogICAgICAgICAgICB3aXRoIGxvZ19wYXRoLm9wZW4oJ2EnLCBidWZmZXJpbmc9MSkgYXMgb3V0cHV0OgogICAgICAgICAgICAgICAgY2hpbGQgPSBzdWJwcm9jZXNzLlBvcGVuKHNwZWNbJ2FyZ3YnXSwgY3dkPXNwZWNbJ3NvdXJjZV9yb290J10sIGVudj1lbnZpcm9ubWVudCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9b3V0cHV0LCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RhcnRfbmV3X3Nlc3Npb249VHJ1ZSkKICAgICAgICAgICAgICAgIHJlY29yZC51cGRhdGUoc3RhdGU9J3J1bm5pbmcnLCB0cmFpbmVyX3BpZD1jaGlsZC5waWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRyYWluZXJfc3RhcnQ9cHJvY2Vzc19pZGVudGl0eShjaGlsZC5waWQpKQogICAgICAgICAgICAgICAgYXRvbWljX2pzb24oc3RhdGVfcGF0aCwgcmVjb3JkKQogICAgICAgICAgICAgICAgcmV0dXJuY29kZSA9IGNoaWxkLndhaXQoKQogICAgICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uOgogICAgICAgICAgICAjIEEgbGl2ZSBjaGlsZCByZW1haW5zIGRpc2NvdmVyYWJsZTsgZG8gbm90IGZhbHNlbHkgcmVjb3JkIHN1Y2Nlc3MuCiAgICAgICAgICAgIHRyYWNlYmFjay5wcmludF9leGMoKQogICAgICAgICAgICByZXR1cm5jb2RlID0gMTI1CiAgICAgICAgcmVjb3JkLnVwZGF0ZShzdGF0ZT0nZXhpdGVkJywgcmV0dXJuY29kZT1yZXR1cm5jb2RlLCBmaW5pc2hlZF91bml4PXRpbWUudGltZSgpKQogICAgICAgIGF0b21pY19qc29uKHN0YXRlX3BhdGgsIHJlY29yZCkKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0td29ya2VyJywgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgd29ya2VyX21haW4ocGFyc2VyLnBhcnNlX2FyZ3MoKS53b3JrZXIpCg==","sha256":"8cb240cd7a71a35b3a45b0c3e813405b1c9c6dc6cb46e6fa424d4e5c5fbf0997"}}
for name, record in EMBEDDED_MODULES.items():
    raw = base64.b64decode(record['base64'])
    if hashlib.sha256(raw).hexdigest() != record['sha256']:
        raise RuntimeError('notebook工具校验失败: ' + name)
    (TOOLS / name).write_bytes(raw)
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
from notebook_state_bc_setup import copy_verified, extract_source, stage_dataset, stage_resume, build_trainer_argv, preflight
from notebook_state_bc import monitor_process, resume_completed, configure_wandb, log_tail
from notebook_state_bc_launch import launch_or_attach, TrainingJob, atomic_json
total_ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
RAM_LIMIT_GIB = round(max(3, min(40, total_ram * .6)), 1)
print('CUDA:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)
print('主机内存保护:', RAM_LIMIT_GIB, 'GiB')


## 代码3 / 6：W&B在线登录

推荐在Colab左侧Secrets保存`WANDB_API_KEY`并允许本notebook访问；也可在下方密码框输入。
不要把密钥写进单元格或聊天。登录/连接失败会明确报错，不会把离线日志冒充在线记录。
页面会打印本次W&B运行链接。重复监控和重连运行时会复用同一个run ID。


In [ ]:
import getpass, uuid
import wandb
from IPython.display import display, Markdown

identity_path = NOTEBOOK_RUN / 'wandb_run.json'
if identity_path.exists():
    wb_identity = json.loads(identity_path.read_text())
    if wb_identity['project'] != WANDB_PROJECT or wb_identity.get('requested_entity') != WANDB_ENTITY:
        raise ValueError('该续训目录已有不同的W&B project/entity；请沿用原设置或另选NOTEBOOK_RUN。')
else:
    wb_identity = {'id': uuid.uuid4().hex[:12], 'project': WANDB_PROJECT,
                   'requested_entity': WANDB_ENTITY, 'name': WANDB_NAME}
    atomic_json(identity_path, wb_identity)

def ensure_wandb():
    current = wandb.run
    if current is not None:
        if current.id != wb_identity['id']:
            raise RuntimeError('已有另一个W&B run；请先结束它再连接本次运行。')
        return current
    if not globals().get('_bc_wandb_logged_in', False):
        key = os.environ.get('WANDB_API_KEY')
        if not key:
            try:
                from google.colab import userdata
                key = userdata.get('WANDB_API_KEY')
            except Exception:
                key = None
        if not key:
            key = getpass.getpass('W&B API Key（输入隐藏）: ').strip()
        if not key:
            raise ValueError('需要W&B登录后再开训，尚未启动trainer。')
        try:
            authenticated = wandb.login(key=key, relogin=True)
        finally:
            del key
        if not authenticated:
            raise RuntimeError('W&B登录未成功，尚未启动trainer。')
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, id=wb_identity['id'],
                     resume='allow', name=wb_identity['name'], mode='online',
                     dir=str(WORK), save_code=False, settings=wandb.Settings(init_timeout=90),
                     config={'decision_seconds': 0.4, 'source_run': str(SOURCE_RUN),
                             'target_total_epochs': TARGET_EPOCHS,
                             'placement_training_only': True, 'placement_backend': 'card_prior_cnn64',
                             'learning_rate': LEARNING_RATE, 'lr_schedule': LR_SCHEDULE,
                             'episode_batch_size': EPISODE_BATCH_SIZE, 'frame_microbatch': FRAME_MICROBATCH})
    configure_wandb(run)
    globals()['_bc_wandb_logged_in'] = True
    return run

wb_run = ensure_wandb()
display(Markdown('W&B：[' + wb_run.url + '](' + wb_run.url + ')'))


## 代码4 / 6：准备源码、全量数据包和父模型

从已发布的CPU数据包索引缓存一次并在`/content`解压。原数据、切分和333个来源分片保持一致。
从最新完整旧BC checkpoint初始化新落点阶段；只迁移旧模型权重，不继承它的Adam、epoch或step。
已存在本阶段`latest.pt`时，控制器自动恢复本阶段；不会重新初始化或再次混入父阶段进度。


In [ ]:
from notebook_state_bc_setup import copy_verified, extract_source
from notebook_state_bc_launch import TrainingJob
marker = WORK / '.notebook/trainer_process.json'
if marker.exists() and TrainingJob(marker).poll() is None:
    raise RuntimeError('本控制器仍在运行，请直接运行代码5重新连接。')
# Refuse concurrent old training on the same GPU; do not kill an active process.
import subprocess
processes = subprocess.run(['ps','-eo','pid,args'],capture_output=True,text=True,check=True).stdout
active = [line for line in processes.splitlines()
          if 'train_state_bc.py' in line and 'ps -eo' not in line]
if active:
    raise RuntimeError('旧trainer仍在运行；请先安全停止并确认checkpoint，再启动新实验。')
if not PACK_INDEX.is_file():
    raise RuntimeError('CPU尚未发布完整包。先运行colab_bc040_pack_cpu.ipynb，再开GPU。')
archive_local = WORK / 'source_placement_joint_gt_20261111.tar.gz'
copy_verified(RELEASE / 'source_placement_joint_gt_20261111.tar.gz', archive_local)
SRC = extract_source(archive_local, WORK/'source_placement_joint36_20261112')
if str(SRC/'src') not in sys.path:
    sys.path.insert(0,str(SRC/'src'))
from cr_rl.data.state_bundle import stage_state_bundle
from tqdm.auto import tqdm
pack_bar = None
pack_phase = None

def bundle_progress(record):
    global pack_bar, pack_phase
    phase = record.get('phase','prepare')
    if phase in ('download','extract'):
        unit = 'B' if phase == 'download' else 'file'
        total = record.get('total_bytes' if phase == 'download' else 'total_files')
        done = record.get('completed_bytes' if phase == 'download' else 'completed_files',0)
        if phase != pack_phase:
            if pack_bar is not None: pack_bar.close()
            pack_bar = tqdm(total=total,desc='数据包下载' if phase=='download' else 'SSD解压',
                            unit=unit,unit_scale=phase=='download')
            pack_phase = phase
        if pack_bar is not None:
            pack_bar.update(max(0,done-pack_bar.n))
    else:
        print('数据包准备:',{k:v for k,v in record.items() if k!='members'},flush=True)

try:
    COLLECTION_ROOT = stage_state_bundle(PACK_INDEX,WORK/'dataset',progress=bundle_progress)
finally:
    if pack_bar is not None: pack_bar.close()
COLLECTION_ROOT = Path(COLLECTION_ROOT)
if not COLLECTION_ROOT.resolve().is_relative_to(WORK.resolve()):
    raise RuntimeError('数据包必须缓存到/content本地SSD。')
print('训练只读本地数据:',COLLECTION_ROOT)
PARENT_STAGE = RELEASE / 'experiment_v04_16/stage_000_be9eead6f870'
PARENT = PARENT_STAGE / 'training/latest.pt'
PARENT_MANIFEST = PARENT_STAGE / 'dataset.manifest.json'
if not PARENT.is_file() or not PARENT_MANIFEST.is_file():
    raise RuntimeError('未找到已确认的完整父checkpoint及数据manifest；不要改用不同模型或不完整文件。')
REGISTRY = COLLECTION_ROOT / 'split_registry.json'
if not REGISTRY.is_file():
    raise RuntimeError('完整数据包缺少split_registry，拒绝改变固定验证划分。')
TRAIN = NOTEBOOK_RUN
COMMAND = [sys.executable,str(SRC/'scripts/run_state_bc_full.py'),
           '--collection-root',str(COLLECTION_ROOT),'--trust-data-files',
           '--require-complete-collection','--expected-total-games',str(EXPECTED_TOTAL_GAMES),
           '--expected-source-shards','333',
           '--out',str(NOTEBOOK_RUN),'--local-work',str(WORK/'training'),
           '--parent-checkpoint',str(PARENT),'--parent-manifest',str(PARENT_MANIFEST),
           '--split-registry',str(REGISTRY),'--min-train-games',str(MIN_TRAIN_GAMES),
           '--validation-fraction',str(VALIDATION_FRACTION),
           '--validation-games',str(VALIDATION_GAMES),
           '--episode-batch-size',str(EPISODE_BATCH_SIZE),
           '--frame-microbatch',str(FRAME_MICROBATCH),'--allow-episode-batch-transition',
           '--epochs',str(TARGET_EPOCHS),'--patience',str(EARLY_STOPPING_PATIENCE),
           '--allow-epoch-extension',
           '--metrics-every-updates','32',
           '--min-delta',str(EARLY_STOPPING_MIN_DELTA),'--learning-rate',str(LEARNING_RATE),
           '--min-learning-rate',str(MIN_LEARNING_RATE),'--warmup-steps',str(WARMUP_STEPS),
           '--upgrade-placement','--placement-cnn-channels','64',
           '--unfreeze-after-checkpoints','4','--unfreeze-learning-rate','1e-5',
           '--ram-limit-gib',str(RAM_LIMIT_GIB),'--device','cuda']
TRAIN_ENV = {'PYTHONPATH':str(SRC/'src'),'OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2',
             'PYTHONDONTWRITEBYTECODE':'1'}
print('先训练落点头；4次定期保存后自动全量解冻，学习率1e-5，保留落点Adam与数据游标。BC只计算标签牌落点，验证和推理计算全部手牌。')
print('验证池4%，每轮随机16局（32视角）；启动后直接训练，不做初始验证。')
print('准备完成，代码5启动；代码6检查数据和checkpoint。')


## 代码5 / 6：启动或重新连接，观察训练和随机验证

只有本单元格启动控制器。重跑会连接已有后台进程；重连后进度读取本阶段记录，不把父阶段step算作新后端训练。
准备、先验统计和首次优化期间会显示阶段信息。每轮完成后显示随机16局验证；W&B同时记录训练和验证指标。


In [ ]:
from notebook_state_bc import JSONLTailer, metric_payload, completed_seats
from tqdm.auto import tqdm
wb_run = ensure_wandb()
status_path = TRAIN/'data_status.json'
waiting = (json.loads(status_path.read_text()).get('phase','').startswith('waiting_')
           if status_path.is_file() else False)
job = launch_or_attach(COMMAND, source_root=SRC, workspace=WORK,
                       log_path=WORK/'training.log',env_overrides=TRAIN_ENV,
                       restart_completed=waiting)
cursor = WORK / '.notebook/full_wandb_cursor.json'
tailer = JSONLTailer(TRAIN/'metrics.jsonl',cursor_path=cursor,identity=wb_run.id)
progress_state_path = WORK / '.notebook/full_progress.json'
progress_state = (json.loads(progress_state_path.read_text())
                  if progress_state_path.is_file() else {})
seats_per_epoch = progress_state.get('seats_per_epoch')
completed = int(progress_state.get('completed_seats',0))
bar = tqdm(total=seats_per_epoch*TARGET_EPOCHS if seats_per_epoch else 0,
           initial=completed,desc='等待准备 / 新后端训练',unit='seat')
prepare_tail = JSONLTailer(WORK/'training.log',cursor_path=WORK/'.notebook/prepare_cursor.json',
                          identity='full-preparation',skip_non_json=True)
validation_bar = None
last_prepare = None
try:
    while True:
        for record, offset in prepare_tail.pending():
            phase = record.get('phase')
            if phase == 'state_bc_prepare':
                detail = {k:v for k,v in record.items() if k not in ['phase','elapsed_seconds']}
                if detail != last_prepare:
                    print('启动准备:',detail,flush=True)
                    last_prepare = detail
                bar.set_description('准备: '+str(record.get('stage',record.get('step','检查数据'))))
            elif phase == 'state_bc_start':
                seats = record.get('train_seats_per_epoch',record.get('train_seats'))
                if seats is not None:
                    seats_per_epoch = int(seats)
                    progress_state['seats_per_epoch'] = seats_per_epoch
                    bar.total = seats_per_epoch*TARGET_EPOCHS
                    bar.refresh()
                bar.set_description('等待第一局训练更新')
            elif phase == 'state_bc_validation_progress':
                done = record.get('episodes_completed',0)//2
                total = record.get('episodes_total',0)//2
                if validation_bar is None or done < validation_bar.n:
                    if validation_bar is not None: validation_bar.close()
                    validation_bar = tqdm(total=total,desc='随机验证',unit='局')
                if total is not None: validation_bar.total=total
                validation_bar.update(max(0,done-validation_bar.n))
            prepare_tail.acknowledge(offset)
        for record, offset in tailer.pending():
            payload = metric_payload(record)
            if payload:
                wb_run.log(payload)
            tailer.acknowledge(offset)
            if record.get('phase') in ('state_bc_train','state_bc_progress'):
                bar.set_description('完整轨迹座位更新')
                if seats_per_epoch:
                    absolute = completed_seats({**record,'phase':'state_bc_train'},seats_per_epoch)
                    bar.update(max(0,absolute-bar.n))
                    progress_state['completed_seats'] = int(bar.n)
                details={'epoch':record.get('epoch'),'step':record.get('optimizer_steps')}
                if 'loss' in record: details['loss']=round(record['loss'],3)
                bar.set_postfix(**details)
            elif record.get('phase') == 'state_bc_validation':
                keys = ['epoch','episodes','loss','slot_accuracy','tile_mean_distance']
                print('16局抽样验证:',{k:record.get(k) for k in keys})
        if progress_state:
            atomic_json(progress_state_path,progress_state)
        status = job.poll()
        if status is not None:
            if status:
                print((WORK/'training.log').read_text(errors='replace')[-6000:])
                raise RuntimeError('训练控制器退出，code='+str(status))
            for name in ['summary.json','data_status.json']:
                p=TRAIN/name
                if p.is_file():print(name,json.loads(p.read_text()))
            break
        time.sleep(2)
except KeyboardInterrupt:
    print('显示已暂停；后台训练保持。重跑代码5重新连接。')
finally:
    bar.close()
    if validation_bar is not None: validation_bar.close()


## 代码6 / 6：本地数据、最优验证与恢复状态
查看本地完整数据回执、训练进度与最优验证。GPU不启动转换器或打包器。


In [ ]:
paths = [COLLECTION_ROOT/'collection.manifest.json', TRAIN/'data_status.json',
         TRAIN/'controller.json', TRAIN/'summary.json']
for p in paths:
    if p.is_file():
        d=json.loads(p.read_text())
        print(p.name,{k:v for k,v in d.items() if k not in ['batches','argv']})
for p in sorted(TRAIN.glob('stage_*/training/best_validation.json')):
    print('最优验证',p,json.loads(p.read_text()))
print('CPU打包索引:',PACK_INDEX)
print('本地完整数据根:',COLLECTION_ROOT)
print('新后端best.pt用于固定旧BC与old对战；latest.pt恢复同阶段。验证尚不等于实战胜率。')
